In [1]:
# installing if needed
%pip install retina-face
%pip install insightface
%pip install onnxruntime
%pip install tqdm

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# imports
import os
import cv2
import numpy as np

from tqdm import tqdm

from retinaface import RetinaFace

from insightface.app import FaceAnalysis

In [5]:
# paths
ROOT_DIR = "naturally_dataset"

GALLERY_DIR = os.path.join(ROOT_DIR, "/Users/admin/Desktop/reliable_rejection_under_degradation/verification_mlp/verification_mlp_multilayer/mlp_multilayer_fullExtendedVersion/natural_dataset/gallery")

PROBE_DIR = os.path.join(ROOT_DIR, "/Users/admin/Desktop/reliable_rejection_under_degradation/verification_mlp/verification_mlp_multilayer/mlp_multilayer_fullExtendedVersion/natural_dataset/probe")

CROPPED_GALLERY_DIR = os.path.join(
    ROOT_DIR,
    "gallery_cropped",
)

GALLERY_EMBED_DIR = os.path.join(
    ROOT_DIR,
    "gallery_embeddings",
)

PROBE_EMBED_DIR = os.path.join(
    ROOT_DIR,
    "probe_embeddings",
)

for folder in [
    CROPPED_GALLERY_DIR,
    GALLERY_EMBED_DIR,
    PROBE_EMBED_DIR,
]:
    os.makedirs(folder, exist_ok=True)

In [6]:
# cropping gallery images using retina face
saved = 0
fallback = 0
failed = 0

for identity in tqdm(os.listdir(GALLERY_DIR)):

    person_dir = os.path.join(
        GALLERY_DIR,
        identity,
    )

    if not os.path.isdir(person_dir):
        continue

    save_person_dir = os.path.join(
        CROPPED_GALLERY_DIR,
        identity,
    )

    os.makedirs(
        save_person_dir,
        exist_ok=True,
    )

    for img_name in os.listdir(person_dir):

        img_path = os.path.join(
            person_dir,
            img_name,
        )

        img = cv2.imread(img_path)

        if img is None:
            failed += 1
            continue

        try:

            faces = RetinaFace.detect_faces(img)

            if isinstance(faces, dict) and len(faces) > 0:

                best_face = max(
                    faces.values(),
                    key=lambda x: (x["facial_area"][2] - x["facial_area"][0])
                    * (x["facial_area"][3] - x["facial_area"][1]),
                )

                x1, y1, x2, y2 = best_face["facial_area"]

                crop = img[
                    y1:y2,
                    x1:x2,
                ]

            else:

                crop = img
                fallback += 1

        except Exception:

            crop = img
            fallback += 1

        cv2.imwrite(
            os.path.join(
                save_person_dir,
                img_name,
            ),
            crop,
        )

        saved += 1

print("=" * 50)
print("Gallery Cropping Complete")
print("=" * 50)
print("Saved :", saved)
print("Fallback :", fallback)
print("Failed :", failed)

100%|██████████| 200/200 [24:01<00:00,  7.21s/it]

Gallery Cropping Complete
Saved : 800
Fallback : 0
Failed : 0


In [7]:
# loading arcface
print("Loading ArcFace...")

app = FaceAnalysis(name="buffalo_l")

app.prepare(
    ctx_id=0,
    det_size=(640, 640),
)

rec_model = app.models["recognition"]

print("ArcFace Ready!")

Loading ArcFace...
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 

In [8]:
# function to generate embeddings
def generate_embeddings(
    image_root,
    embedding_root,
):

    total = 0

    for identity in tqdm(os.listdir(image_root)):

        identity_dir = os.path.join(
            image_root,
            identity,
        )

        if not os.path.isdir(identity_dir):
            continue

        save_dir = os.path.join(
            embedding_root,
            identity,
        )

        os.makedirs(
            save_dir,
            exist_ok=True,
        )

        for img_name in os.listdir(identity_dir):

            img_path = os.path.join(
                identity_dir,
                img_name,
            )

            img = cv2.imread(img_path)

            if img is None:
                continue

            img = cv2.resize(
                img,
                (112, 112),
            )

            img = cv2.cvtColor(
                img,
                cv2.COLOR_BGR2RGB,
            )

            embedding = rec_model.get_feat(img).flatten().astype(np.float32)

            save_path = os.path.join(
                save_dir,
                os.path.splitext(img_name)[0] + ".npy",
            )

            np.save(
                save_path,
                embedding,
            )

            total += 1

    print(f"Saved {total} embeddings.")

In [9]:
# gallery embeddings
generate_embeddings(
    CROPPED_GALLERY_DIR,
    GALLERY_EMBED_DIR,
)

100%|██████████| 200/200 [01:26<00:00,  2.31it/s]

Saved 800 embeddings.


In [10]:
# probe embeddings
generate_embeddings(
    PROBE_DIR,
    PROBE_EMBED_DIR,
)

100%|██████████| 200/200 [00:20<00:00,  9.83it/s]

Saved 200 embeddings.


In [11]:
# verifying the embedding size
sample_identity = os.listdir(GALLERY_EMBED_DIR)[0]

sample_embedding = np.load(
    os.path.join(
        GALLERY_EMBED_DIR,
        sample_identity,
        os.listdir(
            os.path.join(
                GALLERY_EMBED_DIR,
                sample_identity,
            )
        )[0],
    )
)

print(sample_embedding.shape)
print(sample_embedding.dtype)

(512,)
float32
